# Predictor prerequisite — base-model contrastive activations (BLINDED)

Exp1 for the sandboxed/blinded spillover predictor. **Pre-SFT, base-model only.**

- Exp1-A: `[system: plus|minus] + [user]`, prompt-only, last-token AND mean-pooled-over-user-tokens, layers {12,14,16,20}, 14 bipolar evals.
- Exp1-B: `[user] + [assistant: meta pole-exemplar]`, last token, same 14 evals.
- Ethical-framework 3-direction side-check (deon/util/virtue).

**Blinding:** only base `unsloth/llama-3.1-8b-instruct` weights (non-gated mirror of the base instruct model, no fine-tuning); inputs read only from `shared/evals_orthogonalized/`; no checkpoints / adapters / results. A leakage check withholds the bundle if any artifact embeds a fine-tuned/results path.

Run with `?cacheRefresh=true` appended to the Colab URL so the latest committed code loads.

## Setup

In [ ]:
import os
if 'COLAB_GPU' in os.environ or 'COLAB_RELEASE_TAG' in os.environ:
    !git clone https://github.com/nielsrolf/spar-ood-propensities /content/repo 2>/dev/null || !git -C /content/repo pull
    %cd /content/repo/june/predictor_prereq
    # Minimal deps only. Deliberately NOT installing niels/propensities so the
    # niels EvalConfig auto-discovery can never be imported (blinding firewall).
    !pip install -q transformers torch pyyaml tqdm numpy
    from google.colab import drive
    drive.mount('/content/drive')
    _drive_path = '/content/drive/MyDrive/spar-ood-propensities/june/predictor_prereq/outputs'
    os.makedirs(_drive_path, exist_ok=True)
    !ln -sfn {_drive_path} outputs
    print(f'  outputs -> {_drive_path}')
else:
    from pathlib import Path
    %cd {str(Path.cwd())}
    os.makedirs('outputs', exist_ok=True)

In [ ]:
import sys
from pathlib import Path
# unsloth/llama-3.1-8b-instruct is non-gated; HF_TOKEN optional but harmless.
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
except Exception:
    from dotenv import load_dotenv; load_dotenv()
for _p in ['.', str(Path('..').resolve() / 'steering_independence')]:
    if _p not in sys.path:
        sys.path.insert(0, _p)

## Blinding assertions (must pass before any model load)

In [ ]:
import eval_loader as EL
from extract import BASE_MODEL_ID, LAYERS, ALLOWED_BASE_IDS

# Base-only allowlist (unsloth/... is a non-gated mirror of the base instruct
# weights, NOT a fine-tune). Reject anything that looks fine-tuned.
assert BASE_MODEL_ID in ALLOWED_BASE_IDS, BASE_MODEL_ID
_ml = BASE_MODEL_ID.lower()
assert 'instruct' in _ml and not any(
    t in _ml for t in ('sft', 'lora', 'ft-', '-ft', 'checkpoint', 'sweep')
), BASE_MODEL_ID
assert EL.EVALS_ROOT.name == 'evals_orthogonalized' and EL.EVALS_ROOT.is_dir()
# No niels EvalConfig anywhere on the import graph.
assert 'experiments.eval_config' not in sys.modules
assert all((EL.EVALS_ROOT / ev).is_dir() for ev in EL.BIPOLAR_EVALS)
print('Blinding pre-checks OK:')
print(f'  model      = {BASE_MODEL_ID} (base instruct mirror, no checkpoints)')
print(f'  evals_root = {EL.EVALS_ROOT}')
print(f'  layers     = {list(LAYERS)}  | evals = {len(EL.BIPOLAR_EVALS)}')

## Run extraction + build bundle (~10–20 min on A100)

In [ ]:
from build_matrices import build

bundle = build('outputs')  # writes CSVs + bundle.json + PREDICTOR_BUNDLE.md to Drive

In [ ]:
print('Sanity:')
for k, v in bundle['sanity'].items():
    print(f'  {k}: {v:.4f}')
print('\nEthical side-check (expect deon & virtue same side vs util):')
for ly in bundle['layers']:
    c = bundle['ethical_side_check']['by_layer'][ly]
    print(f'  L{ly}: dd-du={c[0][1]:+.3f} dd-uv={c[0][2]:+.3f} dv-uv={c[1][2]:+.3f}')
print('\n--- PREDICTOR_BUNDLE.md ---')
print((Path('outputs') / 'PREDICTOR_BUNDLE.md').read_text())

## Deliverable

Everything for the predictor is under the Drive folder symlinked at `outputs/`:
`A_lasttoken_L*`, `A_meanpooled_L*`, `B_L*` (cosine + gram), `ethical3x3_L*`, `bundle.json`, `PREDICTOR_BUNDLE.md`. The leakage check in `build()` already asserted none of these embed a fine-tuned/results path.